In [1]:
%load_ext autoreload
%autoreload 2
import pm4py
from alignment import *
from processtree import *
import pandas as pd
import random
from tqdm import tqdm
from derivation import *

In [2]:
a = Activity(None, 'a', 100000)
a.id = "4"
c = Activity(None, 'c', 100000)
c.id = "8"
d = Activity(None, 'd', 100000)
d.id = "9"
e = Activity(None, 'e', 100000)
e.id = "6"
f = Activity(None, 'f', 100000)
f.id = "7"

choice = Xor(None, [c,d])
c.set_parent(choice)
d.set_parent(choice)
choice.id = "5"

sequence = Sequence(None, [a, choice])
a.set_parent(sequence)
choice.set_parent(sequence)
sequence.id = "2"

loop = Loop(None, [e,f])
e.set_parent(loop)
f.set_parent(loop)
loop.id = "3"

tree = Sequence(None, [sequence,loop])
sequence.set_parent(tree)
loop.set_parent(tree)
tree.id = "1"
tree

→
  →
    Act( a )
    ×
      Act( c )
      Act( d )
  ↺
    Act( e )
    Act( f )

In [3]:
log = pd.DataFrame({
    'case:concept:name': [1,2,2,2,3,3,3],
    'concept:name': ['b','a','f','e','a','c','e'],
    'time:timestamp': [pd.Timestamp(year=1000+i, month=1, day=1) for i in range(7)]
})

In [4]:
log

,case:concept:name,concept:name,time:timestamp
0,1,b,1000-01-01 00:00:00
1,2,a,1001-01-01 00:00:00
2,2,f,1002-01-01 00:00:00
3,2,e,1003-01-01 00:00:00
4,3,a,1004-01-01 00:00:00
5,3,c,1005-01-01 00:00:00
6,3,e,1006-01-01 00:00:00


In [5]:
model_dist = {('4','8','6','7','6'):0.1, ('4','9','6','7','6'):0.1, ('4','8','6'):0.3, ('4','9','6'):0.3}
log_dist = {('b',): 0.1,
 ('a', 'f', 'e'): 0.2,
 ('a', 'c', 'e'): 0.7}

In [6]:
derivation = DerivationPipeline(tree, log, pl=log_dist, pn_measure=model_dist)

In [7]:
derivation.compute("./example")

1/4	Computing all optimal skip alignments in normal form.
Futures created


100%|██████████| 3/3 [00:05<00:00,  1.78s/it]


Number of computed optimal skip alignments in normal form: 4
Timeouts: 0
2/4	Computing optimal skip alignments in normal form probabilities.


100%|██████████| 3/3 [00:00<?, ?it/s]


Compression in skip alignments was 1 : 4.0
{'c': '8', 'a': '4', 'e': '6', 'f': '7', None: None, 'd': '9'}


100%|██████████| 3/3 [00:00<?, ?it/s]

3/4	Computing skips from all optimal skip alignments in normal form.
4/4	Computing skip probabilities.
Done.


In [8]:
print(derivation.print_blinded())

→, 0.1
  →, 0.0
    Act( a ), 0.0
    ×, 0.2222222222222223
      Act( c ), 0.0
      Act( d ), 0.0
  ↺, 0.0
    Act( e ), 0.027777777777777787
    Act( f ), 0.0


In [9]:
derivation.stats()

---=== Skip alignment computation (timeout = 600s) ===---
Avg. number of sagns per trace variant [incl timeouts]: 1.3333333333333333
Total number of sagns [incl timeouts]: 4
Avg. time per log trace variant (ns) [no timeouts]: 0.0
Total time all sagns (ns) [no timeouts]: 0
---=== Unfolding skip alignments ===---
Avg. number of agns for a trace variant [incl timeouts] (ns): 5.0
Total number of agns [incl timeouts] (ns): 15
Avg. time per set of agns for a skip alignment (ns): 0.0
Total time for all agns (ns): 0
---=== Ebi calls for model paths ===---
Avg. time per model path (ns): -1
Total time for all model paths (ns): -1
---=== Derivation skip probabilities ===---
Avg. time per log trace variant (ns): 0.0
Total time all log trace variant (ns): 0
